# Dat200 CA5

## Group 3 Members

- Vincius Millan
- Stig Espen Paulsberg
- Zaur Kambiev

Kaggle team `Group 3`

# Imports

In [193]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.compose            import ColumnTransformer
from sklearn.preprocessing      import OneHotEncoder, PolynomialFeatures
from sklearn.preprocessing      import OrdinalEncoder
from sklearn.preprocessing      import FunctionTransformer
from sklearn.preprocessing      import StandardScaler
from sklearn.pipeline           import Pipeline
from sklearn.ensemble           import RandomForestClassifier
from sklearn.model_selection    import train_test_split
from sklearn.model_selection    import RandomizedSearchCV, GridSearchCV
from sklearn.model_selection    import StratifiedKFold, KFold
from sklearn.metrics            import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model       import RANSACRegressor, Ridge
from sklearn.impute             import SimpleImputer
from sklearn.cross_decomposition import PLSRegression
from sklearn.ensemble import RandomForestRegressor

# Reading data

In [194]:
df_train = pd.read_csv("dat-200-ca-5-h-2026/train.csv", index_col="id")

# Data exploration

# Data visualization

# Data cleaning

### Remove duplicated values

In [195]:
df_train = df_train.drop_duplicates()
df_train.info()

<class 'pandas.DataFrame'>
Index: 350000 entries, 193355 to 186708
Data columns (total 16 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   age                           329197 non-null  float64
 1   sleep_duration_hrs            324108 non-null  float64
 2   sleep_quality_score           340685 non-null  float64
 3   cardio_mins_wk                338447 non-null  float64
 4   water_intake_liters           341900 non-null  float64
 5   screen_time_recreational_hrs  322694 non-null  float64
 6   social_media_hrs              344953 non-null  float64
 7   blood_pressure_sys            340790 non-null  float64
 8   diet_ultra_processed_pct      337333 non-null  float64
 9   diet_omega3_freq              328543 non-null  float64
 10  smoking_status                325888 non-null  str    
 11  alcohol_drinks_wk             330400 non-null  float64
 12  novel_learning_hrs_wk         324581 non-null  float64


In [196]:
df_train.head(20) 

,age,sleep_duration_hrs,sleep_quality_score,cardio_mins_wk,water_intake_liters,screen_time_recreational_hrs,social_media_hrs,blood_pressure_sys,diet_ultra_processed_pct,diet_omega3_freq,smoking_status,alcohol_drinks_wk,novel_learning_hrs_wk,social_connection_score,chronic_stress_score,target
id,,,,,,,,,,,,,,,,
193355,55.0,NaN,9.0,20.0,0.900000,3.7,1.1,129.0,47.7,1.0,Never,3.0,1.8,8.0,8.0,27
509850,23.0,6.8,8.0,66.0,-3.634081,NaN,1.8,108.0,29.9,7.0,Never,63.0,0.9,5.0,5.0,28
413954,70.0,9.4,9.0,13.0,2.100000,9.5,5.0,137.0,NaN,2.0,Never,1.0,2.4,8.0,5.0,25
94859,40.0,7.5,10.0,200.0,1.600000,5.0,0.5,124.0,NaN,0.0,Never,7.0,0.9,NaN,8.0,29
18214,18.0,6.9,6.0,81.0,NaN,3.6,1.2,118.0,43.2,2.0,Never,0.0,0.5,5.0,10.0,26
287030,36.0,5.4,7.0,25.0,2.000000,3.5,2.0,149.0,36.1,0.0,Current,9.0,2.0,NaN,NaN,26
469491,NaN,7.0,7.0,56.0,1.800000,2.0,1.5,104.0,11.9,1.0,Never,0.0,1.8,6.0,6.0,28
148357,21.0,NaN,6.0,74.0,2.100000,7.3,3.2,123.0,26.6,1.0,Never,1.0,1.5,10.0,7.0,29
242385,63.0,8.8,9.0,13.0,1.800000,6.2,4.8,131.0,32.4,2.0,Never,1.0,2.0,8.0,1.0,30


In [197]:
df_train.describe()

,age,sleep_duration_hrs,sleep_quality_score,cardio_mins_wk,water_intake_liters,screen_time_recreational_hrs,social_media_hrs,blood_pressure_sys,diet_ultra_processed_pct,diet_omega3_freq,alcohol_drinks_wk,novel_learning_hrs_wk,social_connection_score,chronic_stress_score,target
count,329197.000000,324108.000000,340685.000000,338447.000000,341900.000000,322694.000000,344953.000000,340790.000000,337333.000000,328543.000000,330400.000000,324581.000000,326334.000000,332873.000000,350000.000000
mean,44.729375,7.101212,8.259284,98.813542,2.200758,5.509927,2.758805,126.224050,50.017328,1.862280,3.081380,2.268215,6.945810,5.970989,27.816571
std,17.182279,1.291619,1.922099,138.746100,0.715779,2.931575,2.175714,13.174967,23.597665,1.877825,6.177412,2.504443,2.275305,2.795315,1.801970
min,-129.000000,-5.934089,-11.000000,-1300.000000,-5.009674,-24.112509,-19.159991,-7.000000,-187.354820,-17.000000,-59.000000,-23.025042,-16.000000,-22.000000,17.000000
25%,34.000000,6.400000,7.000000,24.000000,1.800000,3.800000,1.300000,119.000000,36.500000,1.000000,0.000000,1.000000,6.000000,4.000000,27.000000
50%,44.000000,7.100000,8.000000,54.000000,2.200000,5.500000,2.400000,126.000000,50.000000,2.000000,1.000000,1.600000,7.000000,6.000000,28.000000
75%,55.000000,7.800000,10.000000,122.000000,2.600000,7.200000,3.900000,134.000000,63.600000,3.000000,4.000000,2.800000,8.000000,8.000000,29.000000
max,218.000000,20.139134,28.000000,1497.000000,9.402479,35.090503,24.661864,259.000000,287.389128,21.000000,65.000000,27.562680,30.000000,34.000000,30.000000


## Preprocessing

### Create dictionairy for filtering

In [198]:
bounds = {
    'age': (18, 90),                            # Age between 18 and 90 according to assignment
    'sleep_duration_hrs': (2, 13),              # Can't sleep negative. Set upper limit at 12 hrs
    'sleep_quality_score': (1, 10),             # Self-rated 1-10
    'cardio_mins_wk': (0, 600),                 # Can't be negative. Cutoff at 600.
    'water_intake_liters': (0, 5.5),              # Can't be negative. Cutoff at 5 liters.
    'screen_time_recreational_hrs': (0, 18),    # Can't be negative. Cutoff at 17 hours.
    'social_media_hrs': (0, 18),                # Same as screen time
    'blood_pressure_sys': (85, 185),            # Reasonable BP range (mmHg)
    'diet_ultra_processed_pct': (0, 100),       # Percentage
    'diet_omega3_freq': (0, 7),                 # Days per week
    'alcohol_drinks_wk': (0, 40),              # Can't be negative. Decided to cut off at 40 because we saw a jump in entries at around 40. 
    'novel_learning_hrs_wk': (0, 20),          # Can't be negative. Decided tp cit pff at 20 because we saw a high amount of entries at that point. 
    'social_connection_score': (1, 10),         # Self-rated 1-10
    'chronic_stress_score': (1, 10),            # Self-rated 1-10
    #'target': (0, None)                         # Keep all target values
}

### Replace out of boundaries values function

In [199]:
def replace_invalid_values(df: pd.DataFrame, valid_ranges: dict) -> pd.DataFrame:
    """Replace values outside the specified valid ranges with NaN."""
    df = df.copy()

    for column, (lower, upper) in valid_ranges.items():

        if lower is not None:
            df.loc[df[column] < lower, column] = np.nan

        if upper is not None:
            df.loc[df[column] > upper, column] = np.nan

    return df

df_train = replace_invalid_values(df_train, bounds)

In [200]:
X = df_train.drop(columns='target')
y = df_train['target']

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    shuffle=True,
    stratify=y
)

### Numerical and Categorical Features Split

In [201]:
nominal_features = [
    'smoking_status'
]

ordinal_features = [
    'sleep_quality_score',
    'social_connection_score', 
    'chronic_stress_score'
]

numerical_features = [
    'age', 
    'sleep_duration_hrs', 
    'cardio_mins_wk',
    'water_intake_liters', 
    'screen_time_recreational_hrs',
    'social_media_hrs', 
    'blood_pressure_sys', 
    'diet_ultra_processed_pct',
    'diet_omega3_freq', 
    'alcohol_drinks_wk',
    'novel_learning_hrs_wk'
]



# Build Pipelines

In [202]:
# Numerical Pipeline
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
    ('scaler', StandardScaler())
])

numeric_poly_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('poly', PolynomialFeatures(
        degree=3,
        include_bias=False
    )),
    ('scaler', StandardScaler())
])

# Ordinal Pipeline
ord_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent', add_indicator=True)),
    ('scaler', StandardScaler())
])

# Nominal Pipeline
nom_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(categories=[['Never', 'Former', 'Current']]))
])

# Combining the pipelines with ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_pipeline, numerical_features),
    ('ord', ord_pipeline, ordinal_features),
    ('nom', nom_pipeline, nominal_features)
])

poly_preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_poly_pipeline, numerical_features),
    ('ord', ord_pipeline, ordinal_features),
    ('nom', nom_pipeline, nominal_features)
])




# Evaluate function

In [203]:
def evaluate_regression(
    model,
    X_train,
    X_test,
    y_train,
    y_test
):
    """Evaluate a regression model on training and test data."""

    # Generate predictions
    y_train_pred = np.asarray(model.predict(X_train)).ravel()
    y_test_pred = np.asarray(model.predict(X_test)).ravel()

    # Calculate test metrics
    mae = mean_absolute_error(y_test, y_test_pred)
    mse = mean_squared_error(y_test, y_test_pred)
    r2 = r2_score(y_test, y_test_pred)

    # Compare training and test MAE
    mae_train = mean_absolute_error(y_train, y_train_pred)
    mae_test = mean_absolute_error(y_test, y_test_pred)
    mae_difference = mae_test - mae_train

    return {
        'MAE': mae,
        'MSE': mse,
        'R²': r2,
        'Training MAE': mae_train,
        'Test MAE': mae_test,
        'MAE difference': mae_difference
    }

# Pipeline A
Polynomial Features + Ridge

In [227]:
# Define the pipeline
poly_ridge_pipeline = Pipeline(steps=[
    ('preprocessor', poly_preprocessor),
    ('model', Ridge())
])

In [228]:
# Define the hyperparameter grid
param_grid = {
    'model__alpha': [0.001, 0.01, 0.1, 1, 10, 100, 500]
}

# Configure GridSearchCV
cv = KFold(n_splits=5, shuffle=True, random_state=42)

poly_ridge_search = GridSearchCV(
    estimator=poly_ridge_pipeline,
    param_grid=param_grid,
    scoring='neg_mean_absolute_error',
    cv=cv,
    n_jobs=1,
    verbose=2
)

poly_ridge_search.fit(X_train, y_train)

Fitting 5 folds for each of 7 candidates, totalling 35 fits
[CV] END .................................model__alpha=0.001; total time=   1.8s
[CV] END .................................model__alpha=0.001; total time=   1.8s
[CV] END .................................model__alpha=0.001; total time=   1.8s
[CV] END .................................model__alpha=0.001; total time=   1.8s
[CV] END .................................model__alpha=0.001; total time=   1.8s
[CV] END ..................................model__alpha=0.01; total time=   1.8s
[CV] END ..................................model__alpha=0.01; total time=   1.8s
[CV] END ..................................model__alpha=0.01; total time=   1.8s
[CV] END ..................................model__alpha=0.01; total time=   1.8s
[CV] END ..................................model__alpha=0.01; total time=   1.8s
[CV] END ...................................model__alpha=0.1; total time=   1.7s
[CV] END ...................................model

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...l', Ridge())])"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__alpha': [0.001, 0.01, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_mean_absolute_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",KFold(n_split... shuffle=True)
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",2
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_

In [206]:
# Display the best hyperparameters and CV score
print('Best parameters:', poly_ridge_search.best_params_)
print('Best CV MAE:', -poly_ridge_search.best_score_)

Best parameters: {'model__alpha': 0.1}
Best CV MAE: 0.930385636692711


In [207]:
# Retrieve the best model
best_poly_ridge = poly_ridge_search.best_estimator_

In [ ]:
# Evaluate the model
poly_ridge_results = evaluate_regression(
    best_poly_ridge,
    X_train,
    X_val,
    y_train,
    y_val
)

poly_ridge_results

{'MAE': 0.9292019957630859,
 'MSE': 1.38886232439613,
 'R²': 0.5723756248122855,
 'Training MAE': 0.9289892366340764,
 'Test MAE': 0.9292019957630859,
 'MAE difference': 0.0002127591290095321}

In [209]:
# Define the pipeline
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(
        random_state=42,
        n_jobs=-1
    ))
])

# Define hyperparameter search space
rf_param_grid = {
    'model__n_estimators': [100],
    'model__max_depth': [10, 15],
    'model__min_samples_leaf': [2, 4],
    'model__max_features': [0.75, 1.0]
}

# Perform hyperparameter optimization
rf_random_search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=rf_param_grid,
    n_iter=2,
    scoring='neg_mean_absolute_error',
    cv=cv,
    random_state=42,
    n_jobs=1,
    verbose=2
)

rf_random_search.fit(X_train, y_train)

Fitting 5 folds for each of 2 candidates, totalling 10 fits
[CV] END model__max_depth=10, model__max_features=0.75, model__min_samples_leaf=4, model__n_estimators=100; total time=   6.6s
[CV] END model__max_depth=10, model__max_features=0.75, model__min_samples_leaf=4, model__n_estimators=100; total time=   6.6s
[CV] END model__max_depth=10, model__max_features=0.75, model__min_samples_leaf=4, model__n_estimators=100; total time=   6.4s
[CV] END model__max_depth=10, model__max_features=0.75, model__min_samples_leaf=4, model__n_estimators=100; total time=   6.4s
[CV] END model__max_depth=10, model__max_features=0.75, model__min_samples_leaf=4, model__n_estimators=100; total time=   6.4s
[CV] END model__max_depth=15, model__max_features=0.75, model__min_samples_leaf=4, model__n_estimators=100; total time=   8.4s
[CV] END model__max_depth=15, model__max_features=0.75, model__min_samples_leaf=4, model__n_estimators=100; total time=   8.4s
[CV] END model__max_depth=15, model__max_features=0

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'model__max_depth': [10, 15], 'model__max_features': [0.75, 1.0], 'model__min_samples_leaf': [2, 4], 'model__n_estimators': [100]}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",2
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'neg_mean_absolute_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",KFold(n_split... shuffle=True)
,"verbose verbose: int, default = 0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",2
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, 

In [210]:
# Retrieve the best model
best_rf_mae = rf_random_search.best_estimator_

In [211]:
# Evaluate the best model
rf_results = evaluate_regression(
    best_rf_mae,
    X_train,
    X_val,
    y_train,
    y_val
)

rf_results

{'MAE': 0.9079364732146419,
 'MSE': 1.3655824676600934,
 'R²': 0.5795433865236816,
 'Training MAE': 0.7973158387861654,
 'Test MAE': 0.9079364732146419,
 'MAE difference': 0.11062063442847647}

# Pipeline B

# Pipeline C

Transformer

In [212]:
ensemble_pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('rfc', RandomForestClassifier(
        random_state=42
    ))
])
ensemble_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocess', ...), ('rfc', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('ord', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If th

### Random Forest Classifier Pipeline

"Is the target at 30 or not?"

In [213]:
param_grid_ensemble = {
    'rfc__n_estimators': [10, 25, 50, 100, 150, 200, 300, 400, 500],
    'rfc__max_depth': [None, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'rfc__criterion': ['gini', 'entropy', 'log_loss'],
    'rfc__min_samples_split': [2, 5, 10, 20],
    'rfc__min_samples_leaf': [1, 3, 5, 10, 20],
    'rfc__max_features': ['sqrt', 'log2'],
    'rfc__min_impurity_decrease': np.arange(0.0, 0.51, 0.01)
}

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

rcf_random_search = RandomizedSearchCV(
    n_iter=25,
    estimator=ensemble_pipeline,
    param_distributions=param_grid_ensemble,
    scoring='f1_macro',
    n_jobs=-1,
    cv=skf,
    random_state=42
)

# Filter for when y is max (30)
y_train_is_max = y_train == 30


rcf_random_search.fit(X_train, y_train_is_max)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'rfc__criterion': ['gini', 'entropy', ...], 'rfc__max_depth': [None, 2, ...], 'rfc__max_features': ['sqrt', 'log2'], 'rfc__min_impurity_decrease': array([0. , ..., 0.49, 0.5 ]), ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",25
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'f1_macro'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and

In [214]:
print(f"Best parameters: {rcf_random_search.best_params_}")
print(f"Best score: {rcf_random_search.best_score_:.3f}")

Best parameters: {'rfc__n_estimators': 10, 'rfc__min_samples_split': 2, 'rfc__min_samples_leaf': 10, 'rfc__min_impurity_decrease': np.float64(0.0), 'rfc__max_features': 'sqrt', 'rfc__max_depth': 7, 'rfc__criterion': 'gini'}
Best score: 0.694


### Regression Pipeline

In [215]:
regression_mask = ~y_train_is_max
X_reg_train, y_reg_train = X_train.loc[regression_mask], y_train.loc[regression_mask]

regression_pipeline = Pipeline(
    [
        ('preprocess', preprocessor),
        ('regressor', Ridge(
            random_state=42
        ))
    ]
)

param_grid_regression = {
        'regressor__alpha':    np.logspace(-3, 3, 30),
        'regressor__solver':   ['lsqr', 'sparse_cg', 'sag', 'saga'],
        'regressor__tol':      [1e-6, 1e-5, 1e-4, 1e-3],
        'regressor__max_iter': [500, 1000, 5000]
    }

cv = KFold(n_splits=5, shuffle=True, random_state=42)


# Should consider Gridsearch
reg_random_search = RandomizedSearchCV(
    n_iter=10,
    estimator=regression_pipeline,
    param_distributions=param_grid_regression,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    cv=cv,
    random_state=42
)

reg_random_search.fit(X_reg_train, y_reg_train)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'regressor__alpha': array([1.0000...00000000e+03]), 'regressor__max_iter': [500, 1000, ...], 'regressor__solver': ['lsqr', 'sparse_cg', ...], 'regressor__tol': [1e-06, 1e-05, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'neg_mean_absolute_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",KFold(n_split... shuffle=True)
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",10
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``

In [216]:
print(f"Best parameters: {reg_random_search.best_params_}")
print(f"Best score: {-reg_random_search.best_score_:.3f}")

Best parameters: {'regressor__tol': 1e-05, 'regressor__solver': 'sag', 'regressor__max_iter': 1000, 'regressor__alpha': np.float64(0.002592943797404667)}
Best score: 0.986


# Polynomial regression pipeline

In [235]:
# Define the pipeline
RF_poly_ridge_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('poly', PolynomialFeatures(
        degree=3,
        include_bias=False
    )),
    ('model', Ridge())
])

In [236]:
# Define hyperparameter search space
poly_ridge_param_grid = {
    'model__alpha': [0.001, 0.01, 10, 100]
}

# Perform hyperparameter optimization
RF_poly_ridge_search = GridSearchCV(
    estimator=poly_ridge_pipeline,
    param_grid=poly_ridge_param_grid,
    scoring='neg_mean_absolute_error',
    cv=cv,
    n_jobs=1,
    verbose=2
)

RF_poly_ridge_search.fit(X_reg_train, y_reg_train)

Fitting 5 folds for each of 4 candidates, totalling 20 fits
[CV] END .................................model__alpha=0.001; total time=   1.3s
[CV] END .................................model__alpha=0.001; total time=   1.2s
[CV] END .................................model__alpha=0.001; total time=   1.3s
[CV] END .................................model__alpha=0.001; total time=   1.3s
[CV] END .................................model__alpha=0.001; total time=   1.2s
[CV] END ..................................model__alpha=0.01; total time=   1.3s
[CV] END ..................................model__alpha=0.01; total time=   1.8s
[CV] END ..................................model__alpha=0.01; total time=   1.9s
[CV] END ..................................model__alpha=0.01; total time=   1.8s
[CV] END ..................................model__alpha=0.01; total time=   1.6s
[CV] END ....................................model__alpha=10; total time=   1.3s
[CV] END ....................................mode

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...l', Ridge())])"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__alpha': [0.001, 0.01, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_mean_absolute_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",KFold(n_split... shuffle=True)
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",2
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_

In [237]:
# Retrieve the best model
best_rfpr_mae = RF_poly_ridge_search.best_estimator_

In [238]:
# Evaluate the best model
rfpr_results = evaluate_regression(
    best_rfpr_mae,
    X_train,
    X_val,
    y_train,
    y_val
)

rfpr_results

{'MAE': 1.0088291685336472,
 'MSE': 1.545007118098518,
 'R²': 0.5242993549956714,
 'Training MAE': 1.007428458015959,
 'Test MAE': 1.0088291685336472,
 'MAE difference': 0.0014007105176883083}

# Model comparison and evaluation

# Kaggle submission

In [239]:
# Select the model
final_model = best_rfpr_mae

In [240]:
# Train using all data train.csv
final_model.fit(X, y)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](15,)","['age','sleep_duration_hrs','sleep_quality_score',..., 'novel_learning_hrs_wk','social_connection_score','chronic_stress_score']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,15
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('ord', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit

In [241]:
# Load test.csv file
df_test = pd.read_csv("dat-200-ca-5-h-2026/test.csv", index_col="id")

In [242]:
# Apply the invalid values treatment in the test.csv
df_test = replace_invalid_values(df_test, bounds)

In [243]:
# Generate predictions
test_predictions = final_model.predict(df_test)

In [244]:
# Create the submission DataFrame
submission = pd.DataFrame({
    'id': df_test.index,
    'target': test_predictions
})

# Save the submission file
submission.to_csv("dat-200-ca-5-h-2026/submission_rfpr.csv", index=False)